# HW2: reproducing super-convergence

The experiments of the [HW2 README](https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course/tree/main/hw2) on one T4: ResNet-56 on CIFAR-10 as in Smith & Topin, [Super-Convergence](https://arxiv.org/abs/1708.07120). The README explains the terms and the setup.

Runtime -> Change runtime type -> T4 GPU. The check after the setup cell stops the notebook on a CPU runtime.

On Kaggle's 2 x T4 the runs took 0.46 h (`lr_range_test`), 0.92 h (`clr`) and 7.40 h (`pc_lr`), one T4 takes about twice as long. Colab stops a session after at most 12 hours and deletes its files, so every run cell ends by downloading `results.zip`, and the PC-LR cells pass `--max-hours 11.5`, which stops training before that. A run appends a row to its CSV after every test, so a stopped run keeps its curve so far.

Runs are independent and can go in separate sessions, run the setup cells first. To plot runs from several sessions, unzip the downloads into `hw2/` and run `uv run python plots.py`.

In [ ]:
!git clone --depth 1 https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course.git /content/repo || git -C /content/repo pull
%cd /content/repo/hw2
!pip install -q lightning

In [ ]:
import torch

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
!nvidia-smi -L

In [ ]:
import shutil

from google.colab import files


def download_results() -> None:
    """
    Zip results/ and download it
    """
    shutil.make_archive("results", "zip", base_dir="results")
    files.download("results.zip")

## 1. LR range test (Fig. 2b)

LR grows linearly from 0 to 3 over 5,000 iterations.

In [ ]:
!python train.py lr_range_test
download_results()

## 2. Super-convergence (Fig. 1a, CLR)

One CLR cycle 0.1 -> 3 -> 0.1 over 10,000 iterations, BN MAF 0.95. Paper: 92.4 %.

In [ ]:
!python train.py clr
download_results()

## 3. Super-convergence on 10,000 samples (Table 1, CLR)

The same run on the first 10,000 training images. Paper: 80.6 %.

In [ ]:
!python train.py clr_10k
download_results()

## 4. Typical training (Fig. 1a, PC-LR)

LR 0.35, divided by 10 at iterations 50,000 and 70,000, 80,000 iterations, BN MAF 0.999. Paper: 91.2 %.

In [ ]:
!python train.py pc_lr --max-hours 11.5
download_results()

## 5. Typical training on 10,000 samples (Table 1, PC-LR)

The same run on the first 10,000 training images. Paper: 71.4 %.

In [ ]:
!python train.py pc_lr_10k --max-hours 11.5
download_results()

## Plots

`plots.py` draws the runs in `results/` as the paper's Fig. 1a, 2b and 4a, and next to the authors' Caffe logs of the same runs, then prints final accuracies.

In [ ]:
!python plots.py

In [ ]:
from pathlib import Path

from IPython.display import Image, display

for path in sorted(Path("results/figures").glob("*.png")):
    display(Image(str(path)))

In [ ]:
download_results()